In [1]:
#download from football.uk.co

import requests

def download_file(url: str, filename: str):

    try:
        # Stream the response to handle large files
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Raise error if request failed

        with open(filename, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:  # filter out keep-alive chunks
                    f.write(chunk)

        print(f"Download complete: {filename}")

    except requests.exceptions.RequestException as e:
        print(f"Download failed: {e}")


# Example usage:
if __name__ == "__main__":
    link = "https://www.football-data.co.uk/mmz4281/2526/E0.csv"  # replace with your download link
    save_as = "pl.csv"  # desired filename
    download_file(link, save_as)

Download complete: pl.csv


In [118]:
import pandas as pd
df=pd.read_csv("pl.csv")
df=df[['Date',
 'Time',
 'HomeTeam',
 'AwayTeam',
 'FTHG',
 'FTAG',
 'FTR',
 'HTHG',
 'HTAG',
 'HTR',
 'Referee',
 'HS',
 'AS',
 'HST',
 'AST',
 'HF',
 'AF',
 'HC',
 'AC',
 'HY',
 'AY',
 'HR','AR']]
df = df.rename(columns={
    'Date': 'match_date',
    'Time': 'kickoff_time',
    'HomeTeam': 'home_team',
    'AwayTeam': 'away_team',
    'FTHG': 'home_goals',
    'FTAG': 'away_goals',
    'FTR': 'result',
    'HTHG': 'ht_home_goals',
    'HTAG': 'ht_away_goals',
    'HTR': 'ht_result',
    'HS': 'home_shots',
    'AS': 'away_shots',
    'HST': 'home_shots_on_target',
    'AST': 'away_shots_on_target',
    'HF': 'home_fouls',
    'AF': 'away_fouls',
    'HC': 'home_corners',
    'AC': 'away_corners',
    'HY': 'home_yellow_cards',
    'AY': 'away_yellow_cards',
    'HR': 'home_red_cards',
    'AR': 'away_red_cards',
    'Referee': 'referee'
})

df.head()
df=df.drop(columns=["referee"])
# Convert match_date to YYYY-MM-DD
team_map = {
    "AFC Bournemouth": "Bournemouth",
    "Brighton & Hove Albion": "Brighton",
    "Tottenham Hotspur": "Tottenham",
    "Man United": "Manchester Utd",
    "Man City": "Manchester City",
    "West Ham United": "West Ham",
    "Wolverhampton Wanderers": "Wolves",
    "Leeds": "Leeds United",
    "Nott'm Forest": "Nott'ham Forest",
    "Newcastle": "Newcastle Utd",
    "Arsenal": "Arsenal",
    "Liverpool": "Liverpool",
    "Chelsea": "Chelsea",
    "Brentford": "Brentford",
    "Crystal Palace": "Crystal Palace",
    "Aston Villa": "Aston Villa",
    "Fulham": "Fulham",
    "Everton": "Everton",
    "Southampton": "Southampton",
    "Burnley": "Burnley",
    "Leicester City": "Leicester City",
    "Sunderland": "Sunderland"
}


df["home_team"] = df["home_team"].replace(team_map)
df["away_team"] = df["away_team"].replace(team_map)

df["match_date"] = pd.to_datetime(df["match_date"], dayfirst=True).dt.strftime("%Y-%m-%d")

# Rebuild game_key to match stats_clean
df["game_key"] = df["match_date"] + "_" + df["home_team"] + "_" + df["away_team"]


df.to_csv("pl1.csv")


In [108]:


league_table = fotmob.read_league_table()
league_table



team  MP   W  D   L  GF  GA  GD  \
league             season                                                       
ENG-Premier League 2526                    Arsenal  22  15  5   2  40  14  26   
                   2526            Manchester City  22  13  4   5  45  21  24   
                   2526                Aston Villa  22  13  4   5  33  25   8   
                   2526                  Liverpool  22  10  6   6  33  29   4   
                   2526          Manchester United  22   9  8   5  38  32   6   
                   2526                    Chelsea  22   9  7   6  36  24  12   
                   2526                  Brentford  22  10  3   9  35  30   5   
                   2526           Newcastle United  22   9  6   7  32  27   5   
                   2526                 Sunderland  22   8  9   5  23  23   0   
                   2526                    Everton  22   9  5   8  24  25  -1   
                   2526                     Fulham  22   9  4   9  30  31  -1   
                   2526     Brighton & Hove Albion  22   7  9   6  32  29   3   
                   2526             Crystal Palace  22   7  7   8  23  25  -2   
                   2526          Tottenham Hotspur  22   7  6   9  31  29   2   
                   2526            AFC Bournemouth  22   6  9   7  35  41  -6   
                   2526               Leeds United  22   6  7   9  30  37  -7   
                   2526          Nottingham Forest  22   6  4  12  21  34 -13   
                   2526            West Ham United  22   4  5  13  24  44 -20   
                   2526                    Burnley  22   3  5  14  23  42 -19   
                   2526    Wolverhampton Wanderers  22   1  5  16  15  41 -26   

                           Pts  
league             season       
ENG-Premier League 2526     50  
                   2526     43  
                   2526     43  
                   2526     36  
                   2526     35  
                   2526     34  
                   2526     33  
                   2526     33  
                   2526     33  
                   2526     32  
                   2526     31  
                   2526     30  
                   2526     28  
                   2526     27  
                   2526     27  
                   2526     25  
                   2526     22  
                   2526     17  
                   2526     14  
                   2526      8

In [129]:
import pandas as pd
import soccerdata as sd

# ==========================
# CONFIG
# ==========================
INPUT_CSV = "pl1.csv"     # your dataset
OUTPUT_CSV = "premier_league_matches_filled.csv"

LEAGUE = "ENG-Premier League"
SEASONS = ["2025/2026"]   # change if needed

# ==========================
# LOAD YOUR DATASET
# ==========================
df = pd.read_csv(INPUT_CSV)
print("Loaded dataset shape:", df.shape)

# Ensure possession/xG/attendance columns exist
for col in ["home_possession", "away_possession", "home_xg", "away_xg", "attendance"]:
    if col not in df.columns:
        df[col] = pd.NA

# ==========================
# LOAD FOTMOB DATA
# ==========================
fotmob = sd.FotMob(leagues=LEAGUE, seasons=SEASONS)
print("Fetching schedule from FotMob...")
schedule = fotmob.read_schedule()

print("Fetching team match stats for possession...")
stats = fotmob.read_team_match_stats(stat_type="Top stats").reset_index()

# --- Extract possession ---
pos_cols = [c for c in stats.columns if "possession" in c.lower()]
if not pos_cols:
    raise ValueError("No possession column found in FotMob stats!")
stats = stats.rename(columns={pos_cols[0]: "possession"})
stats_clean = stats.copy()

# --- Extract date and teams ---
stats_clean["date"] = stats_clean["game"].str.split(" ", n=1).str[0]  # YYYY-MM-DD
teams_str = stats_clean["game"].str.split(" ", n=1).str[1]            # "TeamA-TeamB"
stats_clean["home_team"] = teams_str.str.split("-", n=1).str[0].str.strip()
stats_clean["away_team"] = teams_str.str.split("-", n=1).str[1].str.strip()
print(stats_clean["home_team"].unique)
# --- Map team names ---
team_map = {
    "AFC Bournemouth": "Bournemouth",
    "Brighton & Hove Albion": "Brighton",
    "Tottenham Hotspur": "Tottenham",
    "Manchester United": "Manchester Utd",       # normalize to short form
    "Manchester City": "Manchester City",           # normalize to short form
    "West Ham United": "West Ham",
    "Wolverhampton Wanderers": "Wolves",
    "Leeds United": "Leeds United",
    "Nottingham Forest": "Nott'ham Forest",
    "Newcastle United": "Newcastle Utd",
    "Arsenal": "Arsenal",
    "Liverpool": "Liverpool",
    "Chelsea": "Chelsea",
    "Brentford": "Brentford",
    "Crystal Palace": "Crystal Palace",
    "Aston Villa": "Aston Villa",
    "Fulham": "Fulham",
    "Everton": "Everton",
    "Burnley": "Burnley",
    "Sunderland": "Sunderland",
    }
stats_clean["home_team"] = stats_clean["home_team"].replace(team_map)
stats_clean["away_team"] = stats_clean["away_team"].replace(team_map)
stats_clean["team"] = stats_clean["team"].replace(team_map)

# --- Build game_key ---
stats_clean["game_key"] = stats_clean["date"] + "_" + stats_clean["home_team"] + "_" + stats_clean["away_team"]
print(stats_clean["game_key"].to_list())
# --- Split home/away possession ---
home_stats = stats_clean[stats_clean["team"] == stats_clean["home_team"]][["game_key", "possession"]].rename(columns={"possession":"home_possession"})
away_stats = stats_clean[stats_clean["team"] == stats_clean["away_team"]][["game_key", "possession"]].rename(columns={"possession":"away_possession"})
possession_df = pd.merge(home_stats, away_stats, on="game_key", how="inner")

# --- Merge into CSV ---
df = df.drop(columns=["home_possession","away_possession"], errors="ignore")
df = df.merge(possession_df, on="game_key", how="left")

print("✅ Home/away possession merged.")

# ==========================
# FETCH HOME/AWAY XG
# ==========================
print("Fetching team match stats for xG...")
xg_stats = fotmob.read_team_match_stats(stat_type="Expected goals (xG)").reset_index()
xg_cols = [c for c in xg_stats.columns if "expected goals" in c.lower() or "xg" in c.lower()]
if not xg_cols:
    raise ValueError("No xG column found in FotMob stats!")
xg_stats = xg_stats.rename(columns={xg_cols[0]: "xg_value"})

# --- Extract date and teams ---
xg_stats["date"] = xg_stats["game"].str.split(" ", n=1).str[0]
teams_str = xg_stats["game"].str.split(" ", n=1).str[1]
xg_stats["home_team"] = teams_str.str.split("-", n=1).str[0].str.strip()
xg_stats["away_team"] = teams_str.str.split("-", n=1).str[1].str.strip()

# --- Map names ---
xg_stats["home_team"] = xg_stats["home_team"].replace(team_map)
xg_stats["away_team"] = xg_stats["away_team"].replace(team_map)
xg_stats["team"] = xg_stats["team"].replace(team_map)

# --- Build game_key ---
xg_stats["game_key"] = xg_stats["date"] + "_" + xg_stats["home_team"] + "_" + xg_stats["away_team"]

# --- Split home/away xG ---
home_xg = xg_stats[xg_stats["team"] == xg_stats["home_team"]][["game_key","xg_value"]].rename(columns={"xg_value":"home_xg"})
away_xg = xg_stats[xg_stats["team"] == xg_stats["away_team"]][["game_key","xg_value"]].rename(columns={"xg_value":"away_xg"})
xg_df = pd.merge(home_xg, away_xg, on="game_key", how="inner")

# --- Merge into CSV ---
df = df.drop(columns=["home_xg","away_xg"], errors="ignore")
df = df.merge(xg_df, on="game_key", how="left")
print("✅ Home/away xG merged.")

# ==========================
# SAVE FINAL CSV
# ==========================

df.to_csv(OUTPUT_CSV, index=False)
print("✅ All stats saved to:", OUTPUT_CSV)


Loaded dataset shape: (220, 24)


[01/21/26 01:13:14] INFO     Saving cached data to C:\Users\LEGION\soccerdata\data\FotMob            ]8;id=900979;file://C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=974180;file://C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\soccerdata\_common.py#249\249]8;;\

[2026-01-21 01:13:14] INFO     TLSLibrary:_load_library:401 - Successfully loaded TLS library: C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=3673;file://C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=782671;file://C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\tls_requests\models\libraries.py#401\401]8;;\
                             C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\tls_re                 
                             quests\bin\tls-client-xgo-1.13.1-windows-amd64.dll                                    

[2026-01-21 01:13:15] INFO     TLSLibrary:_load_library:401 - Successfully loaded TLS library: C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


...

                    INFO     [217/220] Retrieving game with id=4813593                                ]8;id=619971;file://C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\soccerdata\fotmob.py\fotmob.py]8;;\:]8;id=710127;file://C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\soccerdata\fotmob.py#392\392]8;;\

                    INFO     [218/220] Retrieving game with id=4813585                                ]8;id=960253;file://C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\soccerdata\fotmob.py\fotmob.py]8;;\:]8;id=289226;file://C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\soccerdata\fotmob.py#392\392]8;;\

                    INFO     [219/220] Retrieving game with id=4813594                                ]8;id=957686;file://C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\soccerdata\fotmob.py\fotmob.py]8;;\:]8;id=778034;file://C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\soccerdata\fotmob.py#392\392]8;;\

                    INFO     [220/220] Retrieving game with id=4813586                                ]8;id=204595;file://C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\soccerdata\fotmob.py\fotmob.py]8;;\:]8;id=934863;file://C:\Users\LEGION\AppData\Roaming\Python\Python313\site-packages\soccerdata\fotmob.py#392\392]8;;\

✅ Home/away xG merged.
✅ All stats saved to: premier_league_matches_filled.csv


In [79]:
print(df1.columns.tolist())
print(df2.columns.tolist())

['Unnamed: 0', 'attendance', 'away_corners', 'away_fouls', 'away_goals', 'away_possession', 'away_red_cards', 'away_shots', 'away_shots_on_target', 'away_team', 'away_xg', 'away_yellow_cards', 'home_corners', 'home_fouls', 'home_goals', 'home_possession', 'home_red_cards', 'home_shots', 'home_shots_on_target', 'home_team', 'home_xg', 'home_yellow_cards', 'ht_away_goals', 'ht_home_goals', 'ht_result', 'kickoff_time', 'match_date', 'result', 'season', 'stadium', 'week']
['Unnamed: 0', 'match_date', 'kickoff_time', 'home_team', 'away_team', 'home_goals', 'away_goals', 'result', 'ht_home_goals', 'ht_away_goals', 'ht_result', 'home_shots', 'away_shots', 'home_shots_on_target', 'away_shots_on_target', 'home_fouls', 'away_fouls', 'home_corners', 'away_corners', 'home_yellow_cards', 'away_yellow_cards', 'home_red_cards', 'away_red_cards']


In [20]:
import pandas as pd

# Load datasets
df1 = pd.read_csv("data1.csv")
df2 = pd.read_csv("premier_league_matches_filled.csv")

print("=== BEFORE ALIGNMENT ===")
print(f"df1 columns: {len(df1.columns)}")
print(f"df2 columns: {len(df2.columns)}")

# Find common columns
common_cols = list(set(df1.columns) & set(df2.columns))
print(f"\nCommon columns: {len(common_cols)}")
print(common_cols)

# Find unique columns in each
df1_only = list(set(df1.columns) - set(df2.columns))
df2_only = list(set(df2.columns) - set(df1.columns))

print(f"\nColumns only in df1: {df1_only}")
print(f"Columns only in df2: {df2_only}")

# Keep only common columns in both datasets
df1_aligned = df1[common_cols].copy()
df2_aligned = df2[common_cols].copy()

# Sort columns alphabetically for consistency
common_cols_sorted = sorted(common_cols)
df1_aligned = df1_aligned[common_cols_sorted]
df2_aligned = df2_aligned[common_cols_sorted]

print("\n=== AFTER ALIGNMENT ===")
print(f"df1_aligned columns: {len(df1_aligned.columns)}")
print(f"df2_aligned columns: {len(df2_aligned.columns)}")
print(f"Columns match: {list(df1_aligned.columns) == list(df2_aligned.columns)}")
df1_aligned = df1_aligned.drop(columns=["Unnamed: 0","game_key"])
df2_aligned = df2_aligned.drop(columns=["Unnamed: 0","game_key"])
# Save aligned datasets
df1_aligned.to_csv("data1_aligned.csv", index=False)
df2_aligned.to_csv("premier_league_matches_filled_aligned.csv", index=False)

print("\n✅ Aligned datasets saved!")
print(f"   - data1_aligned.csv ({len(df1_aligned)} rows)")
print(f"   - premier_league_matches_filled_aligned.csv ({len(df2_aligned)} rows)")

# Show final column list
print(f"\nFinal columns ({len(common_cols_sorted)}):")
for i, col in enumerate(common_cols_sorted, 1):
    print(f"  {i}. {col}")

=== BEFORE ALIGNMENT ===
df1 columns: 32
df2 columns: 29

Common columns: 28
['home_fouls', 'game_key', 'home_red_cards', 'attendance', 'away_xg', 'away_shots_on_target', 'home_yellow_cards', 'result', 'away_yellow_cards', 'ht_away_goals', 'away_shots', 'away_red_cards', 'home_shots_on_target', 'Unnamed: 0', 'ht_result', 'ht_home_goals', 'away_goals', 'home_possession', 'home_goals', 'home_shots', 'away_team', 'away_corners', 'home_corners', 'home_xg', 'away_fouls', 'away_possession', 'match_date', 'home_team']

Columns only in df1: ['week', 'season', 'Unnamed: 0.1', 'stadium']
Columns only in df2: ['kickoff_time']

=== AFTER ALIGNMENT ===
df1_aligned columns: 28
df2_aligned columns: 28
Columns match: True

✅ Aligned datasets saved!
   - data1_aligned.csv (1520 rows)
   - premier_league_matches_filled_aligned.csv (220 rows)

Final columns (28):
  1. Unnamed: 0
  2. attendance
  3. away_corners
  4. away_fouls
  5. away_goals
  6. away_possession
  7. away_red_cards
  8. away_shots
  9.

In [21]:
import pandas as pd
import numpy as np

# Load aligned datasets
df1 = pd.read_csv("data1_aligned.csv")
df2 = pd.read_csv("premier_league_matches_filled_aligned.csv")

print("=== BEFORE FILLING ===")
print(f"df1 shape: {df1.shape}")
print(f"df2 shape: {df2.shape}")
print(f"\ndf1 missing values:\n{df1.isna().sum()}")

# Convert match_date to datetime for both
df1['match_date'] = pd.to_datetime(df1['match_date'], errors='coerce')
df2['match_date'] = pd.to_datetime(df2['match_date'], errors='coerce')

# Ensure team names are strings and stripped
df1['home_team'] = df1['home_team'].astype(str).str.strip()
df1['away_team'] = df1['away_team'].astype(str).str.strip()
df2['home_team'] = df2['home_team'].astype(str).str.strip()
df2['away_team'] = df2['away_team'].astype(str).str.strip()

# Create a merge key for matching (home_team + away_team + match_date)
df1['merge_key'] = (df1['home_team'] + '_' + 
                     df1['away_team'] + '_' + 
                     df1['match_date'].dt.strftime('%Y-%m-%d'))

df2['merge_key'] = (df2['home_team'] + '_' + 
                     df2['away_team'] + '_' + 
                     df2['match_date'].dt.strftime('%Y-%m-%d'))

# Set merge_key as index for efficient lookup
df2_indexed = df2.set_index('merge_key')

# Get all columns to fill (exclude merge_key and matching columns)
cols_to_fill = [col for col in df1.columns if col not in ['merge_key', 'home_team', 'away_team', 'match_date']]

print(f"\nColumns to fill: {len(cols_to_fill)}")

# Fill missing values row by row
filled_count = 0
for idx, row in df1.iterrows():
    merge_key = row['merge_key']
    
    # Check if this match exists in df2
    if merge_key in df2_indexed.index:
        df2_row = df2_indexed.loc[merge_key]
        
        # Fill each column if it's NaN in df1 but exists in df2
        for col in cols_to_fill:
            if col in df2_row.index:
                # Only fill if df1 has NaN and df2 has a value
                if pd.isna(df1.at[idx, col]) and pd.notna(df2_row[col]):
                    df1.at[idx, col] = df2_row[col]
                    filled_count += 1

print(f"\nTotal values filled: {filled_count}")

# Drop the temporary merge_key column
df1_filled = df1.drop(columns=['merge_key'])

# Sort by date
df1_filled = df1_filled.sort_values('match_date').reset_index(drop=True)

print("\n=== AFTER FILLING ===")
print(f"df1_filled shape: {df1_filled.shape}")
print(f"\nRemaining missing values:\n{df1_filled.isna().sum()}")

# Show date range
print(f"\nDate range: {df1_filled['match_date'].min()} to {df1_filled['match_date'].max()}")

# Show last 10 matches to verify
print("\nLast 10 matches:")
print(df1_filled[['match_date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'home_xg', 'away_xg']].tail(10))

# Save the filled dataset
df1_filled.to_csv("merged_dataset.csv", index=False)
print("\n✅ Filled dataset saved as 'merged_dataset.csv'!")

=== BEFORE FILLING ===
df1 shape: (1520, 26)
df2 shape: (220, 26)

df1 missing values:
attendance              280
away_corners            280
away_fouls              280
away_goals              280
away_possession         280
away_red_cards          280
away_shots              280
away_shots_on_target    280
away_team                 0
away_xg                 280
away_yellow_cards       280
home_corners            280
home_fouls              280
home_goals              280
home_possession         280
home_red_cards          280
home_shots              280
home_shots_on_target    280
home_team                 0
home_xg                 280
home_yellow_cards       280
ht_away_goals           280
ht_home_goals           280
ht_result               280
match_date                0
result                  280
dtype: int64

Columns to fill: 23

Total values filled: 2640

=== AFTER FILLING ===
df1_filled shape: (1520, 26)

Remaining missing values:
attendance              280
away_corners     